In [8]:
!pip -q install cryptography

In [9]:
from cryptography.hazmat.primitives.ciphers import Cipher, algorithms, modes
from cryptography.hazmat.primitives.padding import PKCS7
import os

In [10]:
BLOCK_SIZE = 16

key = os.urandom(32)
iv = os.urandom(16)
plaintext = b"Padding oracle attacks recover plaintext without the AES key."

padder = PKCS7(128).padder()
padded = padder.update(plaintext) + padder.finalize()

encryptor = Cipher(
    algorithms.AES(key), modes.CBC(iv)
).encryptor()

ciphertext = encryptor.update(padded) + encryptor.finalize()

print("Original ciphertext:", ciphertext.hex())
print("Initialization vector:", iv.hex())


Original ciphertext: 8e917ef7922f61e0f33957a5e6380e074a0263ab37519cc3dea93dd2ba4c9fc4bda3b31a45e1ca46d3798495a732a2fed0a61ad5d8d17230293bade4cae89fcd
Initialization vector: 6c509c5abaf695d02b13997870e52d78


In [11]:
query_count = 0

def padding_oracle(test_ciphertext):
    global query_count
    query_count += 1

    if len(test_ciphertext) < 32 or len(test_ciphertext) % 16:
        return False

    test_iv = test_ciphertext[:16]
    test_ct = test_ciphertext[16:]

    decryptor = Cipher(
        algorithms.AES(key), modes.CBC(test_iv)
    ).decryptor()

    decrypted = decryptor.update(test_ct) + decryptor.finalize()

    unpadder = PKCS7(128).unpadder()

    try:
        unpadder.update(decrypted) + unpadder.finalize()
        return True
    except ValueError:
        return False

In [12]:
def recover_block(target_block):
    crafted = bytearray(16)
    intermediate = bytearray(16)

    for pad_len in range(1, 17):
        idx = 16 - pad_len

        for j in range(idx + 1, 16):
            crafted[j] = intermediate[j] ^ pad_len

        found = False

        for guess in range(256):
            crafted[idx] = guess

            if padding_oracle(bytes(crafted) + target_block):


                if pad_len == 1:
                    confirm = bytearray(crafted)
                    confirm[idx - 1] ^= 1

                    if not padding_oracle(
                        bytes(confirm) + target_block
                    ):
                        continue

                intermediate[idx] = guess ^ pad_len
                found = True
                break

        if not found:
            raise RuntimeError(
                f"Could not recover byte at index {idx}"
            )

    return bytes(intermediate)

In [13]:
def recover_plaintext(iv, ciphertext):
    blocks = [
        ciphertext[i:i + 16]
        for i in range(0, len(ciphertext), 16)
    ]

    recovered = bytearray()
    previous = iv

    for block_number, block in enumerate(blocks, start=1):
        intermediate = recover_block(block)

        # P_i = D_K(C_i) XOR C_(i-1)
        plaintext_block = bytes(
            a ^ b for a, b in zip(intermediate, previous)
        )

        recovered.extend(plaintext_block)
        previous = block

        print(f"Recovered block {block_number}/{len(blocks)}")

    # Remove the original PKCS#7 padding.
    unpadder = PKCS7(128).unpadder()
    return unpadder.update(bytes(recovered)) + unpadder.finalize()


In [14]:
recovered = recover_plaintext(iv, ciphertext)

print("\nOriginal plaintext: ", plaintext.decode())
print("Recovered plaintext:", recovered.decode())
print("Oracle queries:", query_count)
print("Successful recovery:", recovered == plaintext)
print("Original ciphertext:", ciphertext.hex())
print("Initialization vector:", iv.hex())

Recovered block 1/4
Recovered block 2/4
Recovered block 3/4
Recovered block 4/4

Original plaintext:  Padding oracle attacks recover plaintext without the AES key.
Recovered plaintext: Padding oracle attacks recover plaintext without the AES key.
Oracle queries: 8853
Successful recovery: True
Original ciphertext: 8e917ef7922f61e0f33957a5e6380e074a0263ab37519cc3dea93dd2ba4c9fc4bda3b31a45e1ca46d3798495a732a2fed0a61ad5d8d17230293bade4cae89fcd
Initialization vector: 6c509c5abaf695d02b13997870e52d78
